# 🔬 11주차 실습 노트북
## 내 AI가 실수하는 이유를 진단하고, 더 정확하게 고친다

---

**📌 오늘의 실습 흐름**

```
STEP 1  학습 곡선 그리기         → 내 모델 상태 진단
STEP 2  과적합 일부러 만들기      → 눈으로 확인
STEP 3  Dropout 적용 ⭐          → 전후 비교
STEP 4  Early Stopping 적용 ⭐   → 자동 종료 확인
STEP 5  Precision·Recall·F1 ⭐   → 정확도의 한계 확인
STEP 6  다중 분류 ⭐              → 세 가지 동시에 분류!
```

| 조건 | 값 |
|------|----|
| 데이터 | 당뇨 예측 데이터 (10주차 재사용) |
| 기준 모델 | 10주차 모델 B (Dense 16→8→1) |
| 기본 학습률 | 0.01 |
| 기본 에포크 | 100 |

---
**⚠️ 실행 전 확인**: 런타임 → 런타임 유형 변경 → GPU 선택 (선택사항)

---
## ⚙️ STEP 0 — 환경 설정

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (코랩)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'],
               capture_output=True)
matplotlib.rc('font', family='NanumGothic')
matplotlib.rc('axes', unicode_minus=False)
plt.rcParams['figure.dpi'] = 100

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 재현성 시드 고정
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow 버전: {tf.__version__}')
print('환경 설정 완료! ✅')

---
## 📂 데이터 준비 (10주차와 동일)

빠르게 넘기고 오늘 핵심인 **진단·처방·확장**에 집중합니다.

In [ ]:
# ── 데이터 불러오기
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'
columns = ['임신횟수', '혈당', '혈압', '피부두께',
           '인슐린', 'BMI', '유전기능', '나이', '당뇨여부']
df = pd.read_csv(url, names=columns)

X = df.drop('당뇨여부', axis=1).values
y = df['당뇨여부'].values

# ── 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── 정규화
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'훈련 데이터: {X_train_s.shape[0]}명')
print(f'테스트 데이터: {X_test_s.shape[0]}명')
print(f'당뇨 비율: {y_train.mean():.1%} (훈련) / {y_test.mean():.1%} (테스트)')
print('데이터 준비 완료! ✅')

---
## 📊 STEP 1 — 학습 곡선 그리기

10주차 기준 모델 B를 학습하고 **학습 곡선**을 그려봅니다.

**확인 포인트**: 훈련 손실과 검증 손실이 나란히 내려가나요?

세 가지 패턴을 기억하세요:
- ✅ **정상**: 두 선이 나란히 내려간다
- ⚠️ **과소적합**: 둘 다 높고 안 내려온다
- ❌ **과적합**: 파란선↓↓ 빨간선↑

In [ ]:
# ── 10주차 기준 모델 B 재현
def build_model_B():
    model = Sequential([
        Dense(16, activation='relu', input_shape=(8,)),
        Dense(8,  activation='relu'),
        Dense(1,  activation='sigmoid')
    ], name='model_B')
    model.compile(
        optimizer=Adam(learning_rate=0.01),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model_B = build_model_B()

# ── 학습
history_B = model_B.fit(
    X_train_s, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print(f'최종 훈련 정확도: {history_B.history["accuracy"][-1]:.1%}')
print(f'최종 검증 정확도: {history_B.history["val_accuracy"][-1]:.1%}')

In [ ]:
# ── 학습 곡선 시각화
def plot_history(history, title='학습 곡선', color_train='#00B894', color_val='#FF6B6B'):
    """훈련/검증 손실·정확도 곡선을 나란히 그리는 함수"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    epochs = range(1, len(history.history['loss']) + 1)

    # 손실 곡선
    ax1.plot(epochs, history.history['loss'],
             label='훈련 손실', color=color_train, linewidth=2.5)
    ax1.plot(epochs, history.history['val_loss'],
             label='검증 손실', color=color_val, linewidth=2,
             linestyle='--')
    ax1.set_title('손실 (Loss)')
    ax1.set_xlabel('에포크')
    ax1.set_ylabel('손실')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 정확도 곡선
    ax2.plot(epochs,
             [v*100 for v in history.history['accuracy']],
             label='훈련 정확도', color=color_train, linewidth=2.5)
    ax2.plot(epochs,
             [v*100 for v in history.history['val_accuracy']],
             label='검증 정확도', color=color_val, linewidth=2,
             linestyle='--')
    ax2.set_title('정확도 (Accuracy)')
    ax2.set_xlabel('에포크')
    ax2.set_ylabel('정확도 (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_B, title='STEP 1 — 모델 B 학습 곡선')

# 패턴 진단
train_loss_end = history_B.history['loss'][-1]
val_loss_end   = history_B.history['val_loss'][-1]
diff = val_loss_end - train_loss_end

print('\n📋 패턴 진단')
print(f'  최종 훈련 손실: {train_loss_end:.4f}')
print(f'  최종 검증 손실: {val_loss_end:.4f}')
print(f'  차이 (검증-훈련): {diff:.4f}')
if val_loss_end > 0.6:
    print('  → ⚠️  과소적합 의심: 둘 다 손실이 높아요')
elif diff > 0.08:
    print('  → ❌ 과적합 의심: 훈련/검증 격차가 큽니다')
else:
    print('  → ✅ 정상 학습: 두 선이 비슷하게 내려갔어요')

---
## 😱 STEP 2 — 과적합 일부러 만들기

에포크를 **300**으로 크게 늘려서 과적합이 생기는 순간을 눈으로 확인해봅니다.

**예상 결과**: 훈련 손실은 계속 내려가지만, 검증 손실은 어느 순간부터 다시 올라가기 시작할 거예요.

In [ ]:
# 과적합 모델 — 에포크만 300으로 늘리고 나머지는 동일
model_overfit = build_model_B()

history_overfit = model_overfit.fit(
    X_train_s, y_train,
    epochs=300,          # ← 300으로 늘림!
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

plot_history(history_overfit, title='STEP 2 — 과적합 모델 (에포크 300)')

# 과적합 지점 찾기
val_losses = history_overfit.history['val_loss']
min_val_loss_epoch = val_losses.index(min(val_losses)) + 1

print(f'\n📋 과적합 분석')
print(f'  검증 손실 최솟값 에포크: {min_val_loss_epoch}번째')
print(f'  → 이 지점 이후로 과적합이 시작됐어요!')
print(f'  → Early Stopping이 있었다면 여기서 멈췄겠죠 😊')

---
## 💊 STEP 3 — Dropout 적용 ⭐

**`Dropout(0.3)`** 한 줄만 추가해서 과적합이 얼마나 줄어드는지 비교합니다.

- Dropout(0.3) = 학습 중 30%의 뉴런을 랜덤으로 끈다
- 특정 뉴런에 의존하지 못하게 해서 더 일반적인 패턴을 배우게 됩니다

**비교 포인트**: 두 선(훈련/검증)의 격차가 줄었나요?

In [ ]:
# Dropout 적용 모델
model_dropout = Sequential([
    Dense(16, activation='relu', input_shape=(8,)),
    Dropout(0.3),    # ← 은닉층 1 다음에 추가
    Dense(8,  activation='relu'),
    Dropout(0.3),    # ← 은닉층 2 다음에도 추가
    Dense(1,  activation='sigmoid')
], name='model_dropout')

model_dropout.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 300 에포크로 과적합 유도 조건에서 학습
history_dropout = model_dropout.fit(
    X_train_s, y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

plot_history(history_dropout, title='STEP 3 — Dropout 적용 모델 (에포크 300)')

In [ ]:
# ── Dropout 전후 비교 그래프
epochs_range = range(1, 301)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Dropout 전후 검증 손실 비교', fontsize=14, fontweight='bold')

# 왼쪽: Dropout 없음
ax1.plot(epochs_range, history_overfit.history['loss'],
         label='훈련 손실', color='#00B894', lw=2)
ax1.plot(epochs_range, history_overfit.history['val_loss'],
         label='검증 손실', color='#FF6B6B', lw=2, linestyle='--')
ax1.set_title('Dropout 없음 (과적합 발생)')
ax1.set_xlabel('에포크'); ax1.set_ylabel('손실')
ax1.legend(); ax1.grid(True, alpha=0.3)

# 오른쪽: Dropout 있음
ax2.plot(epochs_range, history_dropout.history['loss'],
         label='훈련 손실', color='#00B894', lw=2)
ax2.plot(epochs_range, history_dropout.history['val_loss'],
         label='검증 손실', color='#FF6B6B', lw=2, linestyle='--')
ax2.set_title('Dropout 적용 후')
ax2.set_xlabel('에포크'); ax2.set_ylabel('손실')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 수치 비교
diff_before = (history_overfit.history['val_loss'][-1]
               - history_overfit.history['loss'][-1])
diff_after  = (history_dropout.history['val_loss'][-1]
               - history_dropout.history['loss'][-1])

print('\n📋 Dropout 전후 훈련/검증 손실 격차')
print(f'  Dropout 없음: {diff_before:.4f}')
print(f'  Dropout 적용: {diff_after:.4f}')
if diff_after < diff_before:
    print(f'  → ✅ 격차가 줄었어요! Dropout 효과가 나타났습니다')
else:
    print(f'  → 이 데이터에서는 효과가 크지 않을 수 있어요')

---
## 🛑 STEP 4 — Early Stopping 적용 ⭐

**`EarlyStopping`**을 사용해서 검증 손실이 더 이상 개선되지 않으면 자동으로 멈추게 합니다.

- `patience=10`: 10 에포크 동안 개선이 없으면 종료
- `restore_best_weights=True`: 검증 손실이 가장 낮았던 시점의 가중치로 자동 복원

**확인 포인트**: 300 에포크를 설정했는데 훨씬 일찍 멈추는지 확인해보세요!

In [ ]:
# Early Stopping 설정
early_stop = EarlyStopping(
    monitor='val_loss',        # 검증 손실을 감시
    patience=10,               # 10 에포크 동안 개선 없으면 종료
    restore_best_weights=True  # 최적 가중치 자동 복원
)

model_es = build_model_B()

history_es = model_es.fit(
    X_train_s, y_train,
    epochs=300,                    # 300으로 설정했지만...
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],        # ← Early Stopping 콜백 추가!
    verbose=1
)

actual_epochs = len(history_es.history['loss'])
print(f'\n🛑 Early Stopping 결과')
print(f'  설정 에포크: 300')
print(f'  실제 종료  : {actual_epochs}번째 에포크에서 자동 종료!')
print(f'  최적 에포크: {actual_epochs - 10}번째 (patience=10 이전)')
print(f'  → 최적 가중치로 자동 복원 완료 ✅')

In [ ]:
plot_history(history_es,
             title=f'STEP 4 — Early Stopping ({actual_epochs}번째 에포크에서 종료)')

# 수직선으로 종료 시점 표시
fig, ax = plt.subplots(figsize=(9, 4))
ep_range = range(1, actual_epochs + 1)
ax.plot(ep_range, history_es.history['loss'],
        label='훈련 손실', color='#00B894', lw=2.5)
ax.plot(ep_range, history_es.history['val_loss'],
        label='검증 손실', color='#FF6B6B', lw=2, linestyle='--')
best_ep = actual_epochs - 10
ax.axvline(x=best_ep, color='#6C5CE7', linewidth=1.5,
           linestyle=':', label=f'최적 에포크 ({best_ep}번째)')
ax.axvline(x=actual_epochs, color='#2D3436', linewidth=1.5,
           linestyle=':', label=f'종료 에포크 ({actual_epochs}번째)')
ax.set_title('Early Stopping 시점 확인')
ax.set_xlabel('에포크'); ax.set_ylabel('손실')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📊 STEP 5 — Precision · Recall · F1 출력 ⭐

정확도(Accuracy)만으로는 부족한 이유를 직접 확인합니다.

**핵심 개념 복습**

| 지표 | 수식 | 의미 |
|------|------|------|
| Precision | TP / (TP+FP) | 당뇨라고 했을 때 실제 당뇨인 비율 |
| Recall | TP / (TP+FN) | 실제 당뇨 환자 중 잡아낸 비율 |
| F1-Score | 2×P×R / (P+R) | Precision과 Recall의 조화 평균 |

**의료 AI에서는 Recall이 더 중요해요** — 환자를 놓치는 게 과잉 진단보다 위험하니까요!

In [ ]:
# Early Stopping 모델로 테스트 데이터 예측
y_pred_prob = model_es.predict(X_test_s)
y_pred      = (y_pred_prob > 0.5).astype(int).flatten()

# 정확도
accuracy = (y_pred == y_test).mean()
print(f'정확도(Accuracy): {accuracy:.1%}')
print()

# 성능 리포트
print('─' * 55)
print('         Precision   Recall   F1-Score   Support')
print('─' * 55)
print(classification_report(
    y_test, y_pred,
    target_names=['정상(0)', '당뇨(1)']
))

In [ ]:
# 혼동행렬 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 혼동행렬
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['정상(0)', '당뇨(1)']
)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('혼동행렬 (Confusion Matrix)', fontsize=13)

# Precision·Recall 막대 그래프
report = classification_report(
    y_test, y_pred,
    target_names=['정상', '당뇨'],
    output_dict=True
)
classes   = ['정상', '당뇨']
precision = [report[c]['precision'] for c in classes]
recall    = [report[c]['recall']    for c in classes]
f1        = [report[c]['f1-score']  for c in classes]

x = np.arange(len(classes))
w = 0.25
axes[1].bar(x - w, precision, w, label='Precision', color='#00B894', alpha=0.8)
axes[1].bar(x,     recall,    w, label='Recall',    color='#FF6B6B', alpha=0.8)
axes[1].bar(x + w, f1,        w, label='F1-Score',  color='#6C5CE7', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(classes)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('클래스별 Precision · Recall · F1', fontsize=13)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
    axes[1].text(i-w, p+0.02, f'{p:.2f}', ha='center', fontsize=9)
    axes[1].text(i,   r+0.02, f'{r:.2f}', ha='center', fontsize=9)
    axes[1].text(i+w, f+0.02, f'{f:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# 핵심 해석
diabetes_recall = report['당뇨']['recall']
diabetes_prec   = report['당뇨']['precision']
print('\n📋 핵심 해석')
print(f'  당뇨 Recall   : {diabetes_recall:.1%}')
print(f'  당뇨 Precision: {diabetes_prec:.1%}')
if diabetes_recall < 0.7:
    print('  → ⚠️  Recall이 낮아요!')
    print('     실제 당뇨 환자 중 AI가 놓치는 비율이 높습니다')
    print('     의료 AI에서는 이 지점을 개선하는 게 중요해요')
else:
    print(f'  → ✅ Recall이 비교적 양호해요')
print(f'\n  정확도만 보면: {accuracy:.1%}')
print(f'  당뇨 Recall까지 보면: {diabetes_recall:.1%}')
print('  → 정확도 숫자 하나로는 모델을 다 이해할 수 없어요!')

---
## 🎲 STEP 6 — 다중 분류 모델 만들기 ⭐

지금까지 **당뇨 O/X (이진 분류)** 를 했다면,
이번엔 **정상·경계·위험 (3가지 동시 분류)** 로 확장합니다.

**코드에서 바뀌는 것 딱 세 줄**

| 항목 | 이진 분류 | 다중 분류 |
|------|-----------|----------|
| 출력층 | `Dense(1)` | `Dense(3)` |
| 활성화 함수 | `sigmoid` | `softmax` |
| 손실함수 | `binary_crossentropy` | `categorical_crossentropy` |

In [ ]:
# ── 혈당 기준으로 3단계 위험군 라벨 재분류
# 정상: 혈당 < 100  /  경계: 100~125  /  위험: 126 이상
def glucose_to_3class(row):
    glucose = row['혈당']
    if glucose < 100:
        return 0   # 정상
    elif glucose < 126:
        return 1   # 경계
    else:
        return 2   # 위험

df['위험군'] = df.apply(glucose_to_3class, axis=1)

# 분포 확인
print('📊 위험군 분포')
labels_map = {0: '정상', 1: '경계', 2: '위험'}
for k, v in df['위험군'].value_counts().sort_index().items():
    print(f'  {labels_map[k]} ({k}): {v}명 ({v/len(df):.1%})')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df['위험군'].value_counts().sort_index()
colors = ['#00B894', '#FDCB6E', '#FF6B6B']
axes[0].bar(['정상', '경계', '위험'], counts.values, color=colors, alpha=0.85)
axes[0].set_title('위험군 분포')
axes[0].set_ylabel('인원 수')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=11)

axes[1].scatter(
    df['BMI'], df['혈당'],
    c=df['위험군'].map({0:'#00B894', 1:'#FDCB6E', 2:'#FF6B6B'}),
    alpha=0.6, s=25
)
axes[1].axhline(y=100, color='#FDCB6E', linestyle='--', lw=1.5, label='경계선(100)')
axes[1].axhline(y=126, color='#FF6B6B', linestyle='--', lw=1.5, label='위험선(126)')
axes[1].set_title('BMI vs 혈당 (색: 위험군)')
axes[1].set_xlabel('BMI'); axes[1].set_ylabel('혈당')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 다중 분류를 위한 데이터 준비
X_multi = df.drop(['당뇨여부', '위험군'], axis=1).values
y_multi = df['위험군'].values

# 원-핫 인코딩 (categorical_crossentropy 사용 시 필요)
y_multi_cat = to_categorical(y_multi, num_classes=3)

# 훈련/테스트 분리 및 정규화
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_multi, y_multi_cat,
    test_size=0.2, random_state=42
)

scaler_m = StandardScaler()
X_m_train_s = scaler_m.fit_transform(X_m_train)
X_m_test_s  = scaler_m.transform(X_m_test)

print(f'훈련 데이터: {X_m_train_s.shape[0]}명')
print(f'테스트 데이터: {X_m_test_s.shape[0]}명')
print(f'출력 형태(원-핫): {y_m_train.shape}  ← [정상, 경계, 위험] 3칸')
print('\n원-핫 인코딩 예시:')
print(f'  정상(0) → {to_categorical(0, 3)}')
print(f'  경계(1) → {to_categorical(1, 3)}')
print(f'  위험(2) → {to_categorical(2, 3)}')

In [ ]:
# ── 다중 분류 모델 (이진 분류 모델과 비교해보세요!)
model_multi = Sequential([
    Dense(16, activation='relu', input_shape=(8,)),   # 은닉층 1
    Dense(8,  activation='relu'),                      # 은닉층 2
    Dense(3,  activation='softmax')  # ← 변경! (1→3, sigmoid→softmax)
], name='model_multi')

model_multi.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='categorical_crossentropy',  # ← 변경! (binary→categorical)
    metrics=['accuracy']
)

print('── 다중 분류 모델 구조 ──')
model_multi.summary()
print('\n💡 이진 분류 모델과 비교해보세요:')
print('   바뀐 것: 출력층 Dense(3), softmax, categorical_crossentropy')
print('   그대로 : 은닉층 구조, 학습률, 배치 크기')

In [ ]:
# ── 다중 분류 모델 학습
early_stop_m = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True
)

history_multi = model_multi.fit(
    X_m_train_s, y_m_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_m],
    verbose=0
)

actual_ep = len(history_multi.history['loss'])
final_acc = history_multi.history['val_accuracy'][-1]

print(f'종료 에포크: {actual_ep}번째')
print(f'최종 검증 정확도: {final_acc:.1%}')

plot_history(history_multi, title='STEP 6 — 다중 분류 모델 학습 곡선')

In [ ]:
# ── 다중 분류 결과 평가
y_m_pred_prob = model_multi.predict(X_m_test_s)
y_m_pred      = np.argmax(y_m_pred_prob, axis=1)   # 가장 높은 확률 클래스 선택
y_m_true      = np.argmax(y_m_test, axis=1)

print('📋 다중 분류 성능 리포트')
print('─' * 58)
print(classification_report(
    y_m_true, y_m_pred,
    target_names=['정상(0)', '경계(1)', '위험(2)']
))

In [ ]:
# ── 다중 분류 혼동행렬 + Softmax 확률 예시
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 혼동행렬
cm_m = confusion_matrix(y_m_true, y_m_pred)
disp_m = ConfusionMatrixDisplay(
    confusion_matrix=cm_m,
    display_labels=['정상', '경계', '위험']
)
disp_m.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('다중 분류 혼동행렬', fontsize=13)

# 클래스별 Precision·Recall 비교
report_m = classification_report(
    y_m_true, y_m_pred,
    target_names=['정상', '경계', '위험'],
    output_dict=True
)
classes_m   = ['정상', '경계', '위험']
precision_m = [report_m[c]['precision'] for c in classes_m]
recall_m    = [report_m[c]['recall']    for c in classes_m]
f1_m        = [report_m[c]['f1-score']  for c in classes_m]

x = np.arange(len(classes_m))
w = 0.25
axes[1].bar(x - w, precision_m, w, label='Precision', color='#00B894', alpha=0.8)
axes[1].bar(x,     recall_m,    w, label='Recall',    color='#FF6B6B', alpha=0.8)
axes[1].bar(x + w, f1_m,        w, label='F1-Score',  color='#6C5CE7', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(classes_m)
axes[1].set_ylim(0, 1.15)
axes[1].set_title('클래스별 Precision · Recall · F1', fontsize=13)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

for i, (p, r, f) in enumerate(zip(precision_m, recall_m, f1_m)):
    axes[1].text(i-w, p+0.02, f'{p:.2f}', ha='center', fontsize=8)
    axes[1].text(i,   r+0.02, f'{r:.2f}', ha='center', fontsize=8)
    axes[1].text(i+w, f+0.02, f'{f:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print('\n💡 어떤 클래스가 가장 잘 분류되었나요?')
best_class = classes_m[np.argmax(f1_m)]
hard_class = classes_m[np.argmin(f1_m)]
print(f'   가장 잘 분류: {best_class} (F1: {max(f1_m):.2f})')
print(f'   가장 어려운 클래스: {hard_class} (F1: {min(f1_m):.2f})')

In [ ]:
# ── Softmax 확률 출력 예시 — 실제 환자 데이터 5명
print('📋 Softmax 확률 예측 결과 (테스트 환자 5명)')
print('─' * 60)
print(f'{"번호":<5} {"정상 확률":>10} {"경계 확률":>10} {"위험 확률":>10} {"→ 예측":>8}')
print('─' * 60)

class_names = ['정상', '경계', '위험']
for i in range(min(5, len(y_m_pred_prob))):
    probs    = y_m_pred_prob[i]
    pred_cls = class_names[np.argmax(probs)]
    true_cls = class_names[y_m_true[i]]
    mark     = '✅' if pred_cls == true_cls else '❌'
    print(f'{i+1:<5} {probs[0]:>9.1%} {probs[1]:>10.1%} {probs[2]:>10.1%}  '
          f'{mark} {pred_cls} (실제: {true_cls})')

print('─' * 60)
print('\n💡 세 확률의 합이 항상 100%예요 — 이게 Softmax의 특징!\n'
      '   가장 높은 확률의 클래스가 최종 예측이 됩니다.')

---
## ✅ 오늘 배운 것 — 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| **학습 곡선** | 훈련/검증 손실 그래프로 정상·과소적합·과적합 진단 |
| **과적합** | 훈련↓↓ 검증↑ — 두 선이 벌어질 때 발생 |
| **Dropout** | `Dropout(0.3)` 한 줄로 과적합 완화 |
| **Early Stopping** | 검증 손실이 더 이상 줄지 않으면 자동 종료 |
| **Precision** | 당뇨라고 했을 때 진짜 당뇨인 비율 |
| **Recall** | 실제 당뇨 환자 중 AI가 잡아낸 비율 (의료에서 더 중요!) |
| **F1-Score** | Precision과 Recall의 균형 지표 |
| **다중 분류** | Dense(3)+softmax+categorical_crossentropy — 코드 세 줄 변경 |

---
### 🚀 12주차 예고
이미지를 분류하는 AI를 만들어봐요 → 의료 영상 AI의 기초인 **CNN**!

---
## 🎁 보너스 — 실험해보기

아래 셀을 수정해가며 결과가 어떻게 달라지는지 확인해보세요!

In [ ]:
# ── ✏️ Dropout 비율 실험
# dropout_rate를 0.1, 0.3, 0.5, 0.7로 바꿔보세요!

dropout_rate = 0.3  # ← 여기를 바꿔보세요

model_exp = Sequential([
    Dense(16, activation='relu', input_shape=(8,)),
    Dropout(dropout_rate),
    Dense(8,  activation='relu'),
    Dropout(dropout_rate),
    Dense(1,  activation='sigmoid')
])
model_exp.compile(
    optimizer=Adam(0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_exp = model_exp.fit(
    X_train_s, y_train,
    epochs=200, batch_size=32,
    validation_split=0.2,
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=0
)

val_acc = history_exp.history['val_accuracy'][-1]
ep_done = len(history_exp.history['loss'])

print(f'Dropout({dropout_rate}) 실험 결과')
print(f'  종료 에포크: {ep_done}번째')
print(f'  검증 정확도: {val_acc:.1%}')

plot_history(history_exp, title=f'Dropout({dropout_rate}) 실험')

In [ ]:
# ── ✏️ Early Stopping patience 실험
# patience를 5, 10, 20, 50으로 바꿔보세요!

patience_val = 10  # ← 여기를 바꿔보세요

model_p = build_model_B()
history_p = model_p.fit(
    X_train_s, y_train,
    epochs=300, batch_size=32,
    validation_split=0.2,
    callbacks=[EarlyStopping(
        patience=patience_val,
        restore_best_weights=True
    )],
    verbose=0
)

ep_done = len(history_p.history['loss'])
val_acc = history_p.history['val_accuracy'][-1]

print(f'patience={patience_val} 실험 결과')
print(f'  종료 에포크: {ep_done}번째 / 300')
print(f'  검증 정확도: {val_acc:.1%}')

plot_history(history_p, title=f'Early Stopping (patience={patience_val})')

In [ ]:
# ── ✏️ 분류 임계값 조정으로 Recall 높이기
# threshold를 0.3~0.7 사이에서 바꿔보세요!
# 낮출수록 더 많은 환자를 당뇨로 분류 → Recall↑, Precision↓

threshold = 0.4  # ← 여기를 바꿔보세요 (기본값 0.5)

y_pred_adj = (y_pred_prob > threshold).astype(int).flatten()

report_adj = classification_report(
    y_test, y_pred_adj,
    target_names=['정상', '당뇨'],
    output_dict=True
)

print(f'임계값 = {threshold} 결과')
print('─' * 48)
print(f'                Precision   Recall   F1-Score')
for cls in ['정상', '당뇨']:
    r = report_adj[cls]
    print(f'  {cls}      '
          f'{r["precision"]:>8.2f}   '
          f'{r["recall"]:>6.2f}   '
          f'{r["f1-score"]:>8.2f}')
print('─' * 48)
print(f'  전체 정확도: {report_adj["accuracy"]:.1%}')
print(f'\n💡 임계값을 낮추면:')
print(f'   당뇨 Recall 올라감 → 환자를 더 많이 잡을 수 있어요')
print(f'   단, Precision은 내려가요 → 과잉 진단이 늘어나요')
print(f'   의료 현장에서는 이 트레이드오프를 상황에 맞게 결정해야 해요')